# Lesson 01: Score Matching for Text

In this notebook, we explore **score matching** -- a technique for estimating the gradient of the log-density without knowing the normalizing constant. We will:

1. Build a `ContinuousScoreNet` that predicts noise from noisy embeddings.
2. Show that noise prediction **is** score estimation (the key equivalence).
3. Train with denoising score matching on synthetic embedding data.
4. Discuss the discrete analog (concrete score from SEDD).

In [ ]:
# Install dependencies
%pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Background: What is the Score?

For a probability density $p(x)$, the **score function** is:

$$s(x) = \nabla_x \log p(x)$$

The key advantage: if $p(x) = \exp(f(x)) / Z$, then $\nabla_x \log p(x) = \nabla_x f(x)$ -- the normalizing constant $Z$ vanishes.

### Why does this matter for text?

In Diffusion-LM, tokens are embedded into continuous space. Diffusion adds Gaussian noise, and the model must reverse this process. The score tells the model which direction to move to increase the data probability.

## 2. The Score-Noise Equivalence

The forward diffusion process adds noise:

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

The conditional score is:

$$\nabla_{x_t} \log p(x_t | x_0) = -\frac{\epsilon}{\sqrt{1 - \bar{\alpha}_t}} = -\frac{\epsilon}{\sigma_t}$$

So a model trained to predict $\epsilon$ implicitly learns the score (up to a known factor $-1/\sigma_t$).

In [ ]:
# Import our score matching module
import sys
sys.path.insert(0, "src")
from score_matching import (
    ContinuousScoreNet,
    DenoisingScoreMatchingTrainer,
    linear_beta_schedule,
    compute_alpha_bars,
)

In [ ]:
# Create a small score network
embed_dim = 64
model = ContinuousScoreNet(
    embed_dim=embed_dim,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"ContinuousScoreNet parameters: {total_params:,}")

In [ ]:
# Set up the trainer
num_timesteps = 200
trainer = DenoisingScoreMatchingTrainer(
    model,
    num_timesteps=num_timesteps,
    lr=3e-4,
)
trainer.to(device)

# Visualize the noise schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
alpha_bars = trainer.alpha_bars.cpu().numpy()
axes[0].plot(alpha_bars)
axes[0].set_xlabel("Timestep t")
axes[0].set_ylabel("alpha_bar_t")
axes[0].set_title("Signal retention (alpha_bar)")

sigma_t = np.sqrt(1 - alpha_bars)
axes[1].plot(sigma_t)
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("sigma_t")
axes[1].set_title("Noise level (sigma)")
plt.tight_layout()
plt.show()

## 3. Training with Denoising Score Matching

We train on synthetic embedding data (random clusters in embedding space).

In [ ]:
# Generate synthetic "clean embeddings" -- 5 clusters in embedding space
n_samples = 512
seq_len = 16

n_clusters = 5
centers = torch.randn(n_clusters, embed_dim, device=device) * 3.0

cluster_ids = torch.randint(0, n_clusters, (n_samples, seq_len), device=device)
x_0 = centers[cluster_ids] + torch.randn(n_samples, seq_len, embed_dim, device=device) * 0.1

print(f"Training data shape: {x_0.shape}")

In [ ]:
# Train for 200 steps
losses = []
batch_size = 64

for step in range(200):
    idx = torch.randint(0, n_samples, (batch_size,))
    batch = x_0[idx]
    loss = trainer.train_step(batch)
    losses.append(loss)
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss = {loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Training step")
plt.ylabel("DSM Loss")
plt.title("Denoising Score Matching Training")
plt.show()

## 4. Verifying the Score-Noise Equivalence

In [ ]:
# Pick a batch and a fixed timestep
test_batch = x_0[:8]
t = torch.full((8,), 100, device=device, dtype=torch.long)

# Add noise
x_t, epsilon_true = trainer.add_noise(test_batch, t)

# Path 1: model predicts noise, we convert to score
model.eval()
with torch.no_grad():
    epsilon_pred = model(x_t, t)

alpha_bar_t = trainer.alpha_bars[t[0]]
sigma_t_val = (1 - alpha_bar_t).sqrt()
score_path1 = -epsilon_pred / sigma_t_val

# Path 2: use the trainer's get_score method
score_path2 = trainer.get_score(x_t, t)

max_diff = (score_path1 - score_path2).abs().max().item()
print(f"Max difference between score computation paths: {max_diff:.2e}")
print(f"Confirmed: noise prediction and score estimation are equivalent!")

# Compare predicted vs true score
true_score = -epsilon_true / sigma_t_val
score_error = (score_path2 - true_score).abs().mean().item()
print(f"\nMean score prediction error: {score_error:.4f}")
print(f"(This improves with more training data and steps)")

## 5. Visualizing the Score at Different Noise Levels

In [ ]:
timesteps_to_check = [10, 50, 100, 150, 190]
score_magnitudes = []

for t_val in timesteps_to_check:
    t = torch.full((8,), t_val, device=device, dtype=torch.long)
    x_t, _ = trainer.add_noise(test_batch, t)
    score = trainer.get_score(x_t, t)
    mag = score.norm(dim=-1).mean().item()
    score_magnitudes.append(mag)
    sigma = (1 - trainer.alpha_bars[t_val]).sqrt().item()
    print(f"t={t_val:3d} | sigma_t={sigma:.3f} | mean score magnitude: {mag:.4f}")

plt.figure(figsize=(8, 4))
plt.bar([str(t) for t in timesteps_to_check], score_magnitudes)
plt.xlabel("Timestep")
plt.ylabel("Mean Score Magnitude")
plt.title("Score Magnitude vs Noise Level")
plt.show()

## 6. The Discrete Analog: Concrete Score (SEDD)

For discrete tokens, we cannot take gradients in token space. SEDD defines the **concrete score** as a ratio of transition probabilities:

$$s(x, y, t) = \frac{p(x_t = y \mid x_0)}{p(x_t = x_i \mid x_0)}$$

In [ ]:
from score_matching import concrete_score_example

vocab_size = 10
batch, seq_len_demo = 2, 5

logits = torch.randn(batch, seq_len_demo, vocab_size)
x = torch.randint(0, vocab_size, (batch, seq_len_demo))

scores = concrete_score_example(logits, x, vocab_size)

print(f"Input tokens (batch 0): {x[0].tolist()}")
print(f"\nConcrete scores for position 0 (token {x[0, 0].item()}):")
print(f"  Scores: {scores[0, 0].detach().numpy().round(3)}")
print(f"  Score for current token: {scores[0, 0, x[0, 0]].item():.4f} (should be 0)")
print(f"\nHigher score = model thinks this replacement is more likely.")

## Summary

1. The **score function** avoids the intractable normalizing constant.
2. **Denoising score matching** trains a model to predict noise added during forward diffusion.
3. Noise prediction **is** score estimation: $\text{score} = -\epsilon / \sigma_t$.
4. For discrete tokens, SEDD's **concrete score** replaces gradients with probability ratios.

### Next: Flow Matching

In the next lesson, we explore **flow matching** -- a simpler alternative that replaces the stochastic SDE with a deterministic ODE along straight paths.